# Whisper Pro: Transcripción Robusta de Larga Duración

Este cuaderno utiliza el modelo **Whisper large-v3** para transcribir audios extensos (3+ horas) con máxima precisión. Incluye herramientas para exportar el resultado a archivos de texto y Markdown.

In [ ]:
# 1. Instalación de dependencias actualizadas
!pip install -U openai-whisper
!pip install setuptools-rust
!apt-get install -y ffmpeg

In [ ]:
import whisper
import torch
import os
import datetime

# Configuración de dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

# 2. Cargar el modelo más robusto (large-v3)
print("Cargando modelo Large-v3 (esto puede tardar unos minutos en Kaggle)...")
model = whisper.load_model("large-v3", device=device)
print("Modelo robusto cargado correctamente.")

In [ ]:
def save_transcription(text, filename_base):
    """Guarda la transcripción en formato .txt y .md"""
    # Guardar como TXT
    with open(f"{filename_base}.txt", "w", encoding="utf-8") as f:
        f.write(text)
    
    # Guardar como MD
    with open(f"{filename_base}.md", "w", encoding="utf-8") as f:
        f.write(f"# Transcripción\n\n{text}")
    
    print(f"Archivos guardados: {filename_base}.txt y {filename_base}.md")

def transcribe_audio_robust(path, language=None):
    """Transcribe audios largos con parámetros de estabilidad"""
    if not os.path.exists(path):
        return "Error: Archivo no encontrado."
    
    print(f"Iniciando transcripción de: {path}")
    start_time = datetime.datetime.now()
    
    # Parámetros optimizados para audios largos:
    # verbose=True permite ver el progreso segmento a segmento.
    # beam_size=5 mejora la precisión.
    # fp16=True usa la GPU de forma eficiente.
    result = model.transcribe(
        path, 
        verbose=True, 
        language=language, 
        task="transcribe",
        beam_size=5,
        best_of=5,
        fp16=True
    )
    
    end_time = datetime.datetime.now()
    duration = end_time - start_time
    print(f"\nTranscripción completada en: {duration}")
    
    # Generar nombre de archivo basado en el original
    base_name = os.path.splitext(os.path.basename(path))[0]
    save_transcription(result["text"], base_name)
    
    return result["text"]

# Ejemplo de uso:
# path_to_audio = "/kaggle/input/mi-audio-largo/clase_3_horas.mp3"
# texto_final = transcribe_audio_robust(path_to_audio, language="es")